# Tu copia personal · Aula UTP

Este cuaderno se abre desde GitHub. En Colab elige **Archivo → Guardar una copia en Drive** antes de conservar cambios. Renombra esa copia y ejecuta de arriba abajo. Cada estudiante trabaja en su propio archivo y entorno.

Los ejemplos son didácticos: sigue la sección indicada por Cristian para esta clase. Las comprobaciones son voluntarias, sin entregas ni notas.

# Una foto → objetos detectados → un registro útil
## YOLO explicado para el Diplomado UTP · Cristian Alvarado
**Caso completo:** revisar una escena, localizar objetos, contar por categoría y guardar un registro CSV y una imagen marcada.

Sirve como punto de partida para revisar fotos de un parqueadero, identificar objetos de oficina o contar objetos visibles de categorías conocidas. No es un inventario certificado: los objetos ocultos o confundidos requieren revisión humana.

**Dinámica:** abrir Colab, leer la explicación, ejecutar las celdas y conversar sobre lo que muestran. No hay entregas ni ejercicios obligatorios.

1. Guarda una copia si deseas conservar cambios.
2. Ejecuta las celdas en orden. La demostración usa imágenes incluidas en la biblioteca.
3. La primera instalación y descarga de pesos necesitan internet. CPU es suficiente para estas fotos.
4. Para una foto propia, al final hay una opción de carga desactivada inicialmente.


## Ruta por encuentro · del dibujo a una foto

- **DL · Encuentro 2, 7 de noviembre:** secciones 1–6. Mira la foto, comprueba las categorías del archivo de pesos y sigue una caja hasta su registro. Esto es reutilizar aprendizaje previo; no ajustamos el modelo para otra categoría.
- **DL · Encuentro 4, 14 de noviembre:** vuelve a ejecutar preparación y secciones 3–6 para recuperar foto, modelo y resultados. Después secciones 7–11: conteo, umbral, filtro y exportación. Continúa con **13, evaluación con cajas de referencia**, antes de pasar al bloque 14 del laboratorio DL.
- **Sección 12:** foto propia opcional y desactivada; no es necesaria para ninguno de los encuentros.

**La sección 13 crea sus propias cajas inventadas** para poder comprobar errores. No afirma que las fotografías de demostración estén evaluadas ni produce mAP de ellas. No necesitas el bloque 08 ni SHAP del otro cuaderno.

**Entorno:** CPU suficiente; no hace falta contratar servicios, activar GPU ni usar claves de API. La primera instalación y la descarga de pesos necesitan internet. Si se reinicia Colab, se pierden variables: ejecuta de nuevo las secciones 2–6. Para conservar cambios usa tu copia personal.


### Nombres del modelo, los datos y el dispositivo

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **YOLO** — YOLO: You Only Look Once (Solo se mira una vez). Nombre de una familia de modelos de detección que propone categorías y cajas a partir de imágenes. El nombre nació con un enfoque que calcula esas propuestas en una evaluación de la red. Las variantes tienen arquitecturas y procesamiento propios; una caja conservada no demuestra que la propuesta sea correcta. [Fuente](https://arxiv.org/abs/1506.02640)

- **COCO** — COCO: Common Objects in Context (Objetos comunes en contexto). Conjunto de imágenes con anotaciones de objetos en escenas cotidianas. Sirve para desarrollar y comparar modelos de visión; no es un detector por sí mismo. El modelo usado en este cuaderno ofrece 80 categorías de ese conjunto, no cualquier objeto ni cualquier defecto agrícola. [Fuente](https://arxiv.org/abs/1405.0312)

- **CPU** — CPU: Central Processing Unit (Unidad central de procesamiento). Procesador que ejecuta las instrucciones generales de un programa. En los cuadernos, cpu identifica el dispositivo que realiza los cálculos. No es una categoría aprendida ni una medida de calidad. [Fuente](https://www.intel.com/content/www/us/en/products/docs/processors/cpu-vs-gpu.html)

- **RGB** — RGB: Red, Green, Blue (Rojo, verde y azul). Forma de representar un color mediante tres componentes o canales. Cada canal indica cuánto rojo, verde o azul participa en el color del píxel. La escala depende del formato: en imágenes de ocho bits suele ser 0–255 por canal; una preparación puede convertirla a 0–1. [Fuente](https://www.w3.org/TR/css-color-4/#rgb-functions)


## 1. Decide qué respuesta necesitas

Imagina que llega una foto a un puesto de revisión. Antes de hacer una lista, observa qué aparece y qué necesita saber quien la revisa.

- Si solo necesita saber si la foto contiene vehículos, busca una **etiqueta de toda la imagen**: clasificación.
- Si necesita señalar cada vehículo, busca una categoría y su ubicación: **detección**. Un rectángulo o **caja** rodea cada propuesta.
- Si necesita su contorno, debe marcar qué cuadritos de la imagen pertenecen a él. Esos cuadritos son **píxeles**; el conjunto marcado es una **máscara**: segmentación.

Aquí haremos detección. YOLO es la familia de modelos que usaremos. No es un buscador universal: el archivo de pesos conoce **80 categorías de COCO**, un conjunto de imágenes usado para su aprendizaje previo. No incluye «café sano» o «café defectuoso». Cambiar el nombre de una etiqueta no enseña esas categorías.

**Antes de ejecutar:** cuando aparezca una caja, ¿qué tres cosas revisarás? Qué categoría propone, dónde la ubica y qué puntuación le asigna. Esa puntuación sirve para filtrar propuestas; no garantiza que sean correctas.


## 2. Preparar las herramientas
Una biblioteca reúne funciones. `ultralytics` proporciona YOLO; `pandas` organiza resultados; `matplotlib` dibuja.
La instalación es preparación del entorno: todavía no entrenamos ni predecimos nada. Comprobamos la versión 8.4.175 aunque ya exista otra instalación. Si la biblioteca estaba cargada y se actualiza, será necesario reiniciar el entorno y volver a ejecutar desde esta sección. [Código oficial de esta versión](https://github.com/ultralytics/ultralytics/tree/v8.4.175).


In [ ]:
import sys, subprocess
from importlib import metadata

VERSION_ULTRALYTICS = '8.4.175'
try:
    version_instalada = metadata.version('ultralytics')
except metadata.PackageNotFoundError:
    version_instalada = None
if version_instalada != VERSION_ULTRALYTICS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           f'ultralytics=={VERSION_ULTRALYTICS}'])
import ultralytics
if ultralytics.__version__ != VERSION_ULTRALYTICS:
    raise RuntimeError('Ultralytics sigue cargado con otra versión. Reinicia el entorno y ejecuta de nuevo desde la preparación.')
print('Ultralytics:', ultralytics.__version__)


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, FileLink
import torch
from ultralytics import YOLO
from ultralytics.utils import ASSETS

SALIDA = Path('resultados_yolo')
SALIDA.mkdir(exist_ok=True)
torch.set_num_threads(2)
TRADUCCION = {'person':'persona','bus':'bus','car':'automóvil','truck':'camión',
              'bicycle':'bicicleta','motorcycle':'motocicleta','chair':'silla',
              'bottle':'botella','cup':'taza','laptop':'portátil','cell phone':'teléfono',
              'book':'libro','backpack':'mochila','tie':'corbata'}
def nombre_es(nombre):
    return TRADUCCION.get(nombre, nombre)
print('PyTorch:',torch.__version__)
print('Los nombres no traducidos conservan la categoría original en inglés.')


## 3. Cargar lo que el modelo ya aprendió
`YOLO('yolo26n.pt')` carga pesos preentrenados de YOLO26 nano. «Nano» identifica una variante pequeña.

**Conexión con redes neuronales:** durante un entrenamiento previo se ajustaron filtros y parámetros. Ahora reutilizamos ese aprendizaje para **inferencia**, es decir, producir una predicción. En este cuaderno no ejecutamos retropropagación ni actualizamos pesos.

Los pesos se descargan automáticamente una vez. La siguiente celda muestra ejemplos del catálogo y busca «café» en todas sus categorías. Después prueba «persona»: cambiar la consulta solo revisa nombres disponibles, no enseña nada nuevo.


In [ ]:
modelo = YOLO('yolo26n.pt')
categorias = pd.DataFrame([{'id':i,'clase_original':nombre,'nombre':nombre_es(nombre)}
                           for i,nombre in modelo.names.items()])
print('Categorías conocidas:',len(categorias))
print('Ejemplos del catálogo:',', '.join(categorias.nombre.head(15)))
print('Son solo ejemplos; una categoría no mostrada aún puede existir. La búsqueda consulta el catálogo completo.')
BUSCAR_CATEGORIA='café' # después prueba 'persona' sin cambiar los pesos
consulta=BUSCAR_CATEGORIA.strip().casefold()
coincidencias=categorias[(categorias.nombre.str.casefold().str.contains(consulta,regex=False))|
                         (categorias.clase_original.str.casefold().str.contains(consulta,regex=False))] if consulta else categorias.iloc[:0]
if coincidencias.empty:
    print('No hay coincidencias para',repr(BUSCAR_CATEGORIA),'en este catálogo. No renombres otra clase para fingir que la reconoce.')
else:
    print('Coincidencias en todo el catálogo:',', '.join(coincidencias.nombre))


## 4. Mira la foto antes de pedir una respuesta

Usamos bus.jpg, incluida en la biblioteca. No necesitas subir una foto personal. Observa primero los objetos visibles sin cajas.

Una imagen digital guarda muchos **píxeles**, pequeños cuadritos de color. RGB significa que cada cuadrito se representa con tres valores: rojo, verde y azul. Ancho y alto cuentan cuántos cuadritos tiene en cada dirección. La expresión «matriz de píxeles» solo nombra esa organización en filas y columnas.

**Tip:** nombra dos objetos que esperarías encontrar y alguno difícil, por ejemplo parcialmente fuera del encuadre. No dibujamos aún una respuesta correcta de evaluación: mirar la foto ayuda a revisar, pero no crea automáticamente anotaciones para medir precisión.


In [ ]:
foto = ASSETS / 'bus.jpg'
original = Image.open(foto).convert('RGB')
print('Ancho y alto:',original.size,'· canales: RGB')
fig,ax=plt.subplots(figsize=(5,7))
ax.imshow(original);ax.axis('off');ax.set_title('Entrada: la foto sin predicciones')
plt.tight_layout();plt.savefig(SALIDA/'01_original.png',dpi=130);plt.show()


## 5. Detectar: entender una llamada completa
- `source`: la foto que vamos a revisar.
- `conf=0.25`: conserva propuestas con puntuación estrictamente mayor que 0,25 en esta versión de Ultralytics.
- `imgsz=640`: referencia de tamaño para preparar la entrada; la biblioteca ajusta escala y relleno.
- `device='cpu'`: ejecuta con procesador, sin exigir GPU.

La confianza es una **puntuación del modelo**. No significa que «0,90 garantiza 90 % de aciertos». Para medir calidad necesitamos imágenes etiquetadas y comparar predicción con realidad.


**Detalle de la frontera:** Ultralytics 8.4.175 aplica puntuación > conf, no ≥. Una puntuación exactamente igual al corte se descarta. En algunos esquemas de la web usamos ≥ para enseñar otra regla explícita: siempre hay que revisar la convención de la herramienta. El corte por puntuación no es el criterio de solapamiento de cajas.


In [ ]:
UMBRAL = 0.25
resultado = modelo.predict(source=str(foto), conf=UMBRAL, imgsz=640,
                           device='cpu', verbose=False)[0]
print('Detecciones conservadas:',len(resultado.boxes))
resultado.save(filename=str(SALIDA/'02_foto_detectada.jpg'))
fig,axs=plt.subplots(1,2,figsize=(10,7))
axs[0].imshow(original);axs[0].set_title('Antes: píxeles')
axs[1].imshow(resultado.plot()[:,:,::-1]);axs[1].set_title('Después: categorías, cajas y puntuaciones')
for ax in axs:ax.axis('off')
plt.tight_layout();plt.savefig(SALIDA/'03_antes_despues.png',dpi=130);plt.show()


## 6. De la caja visible al dato guardado

Primero mira un rectángulo de la foto. Para encontrarlo de nuevo necesitamos su categoría, puntuación y posición. La próxima celda guarda **una fila por propuesta**, no una fila por objeto confirmado.

- **Categoría:** nombre que propone el modelo, como persona o bus. boxes.cls guarda su identificador numérico.
- **Puntuación:** número que usamos para filtrar propuestas. Está en boxes.conf; no es IoU ni porcentaje de calidad garantizada.
- **Posición:** cuatro coordenadas en boxes.xyxy. Un par señala la esquina superior izquierda y otro, la inferior derecha. El origen está arriba a la izquierda: hacia la derecha aumenta x y hacia abajo aumenta y.

Ejemplo solo para entender: [10,20,110,220] significa empezar a 10 píxeles del borde izquierdo y 20 del superior; terminar a 110 y 220. El ancho es 110−10=100 y el alto 220−20=200. La tabla siguiente usa **las posiciones que devolvió el modelo**, no esos números inventados.

**Tip:** busca en la foto la primera detección que imprime el código. Comprueba que su categoría y ubicación se puedan rastrear hasta esa misma caja antes de usar el conteo.


In [ ]:
def tabla_detecciones(r):
    columnas=['deteccion','clase_id','clase_original','categoria','puntuacion',
              'x1','y1','x2','y2','ancho_px','alto_px']
    filas=[]
    for i,caja in enumerate(r.boxes, start=1):
        clase=int(caja.cls.item());nombre=r.names[clase]
        x1,y1,x2,y2=caja.xyxy[0].cpu().numpy().tolist()
        filas.append([i,clase,nombre,nombre_es(nombre),float(caja.conf.item()),
                      x1,y1,x2,y2,x2-x1,y2-y1])
    return pd.DataFrame(filas,columns=columnas)

tabla = tabla_detecciones(resultado)
display(tabla.round(3))
assert len(tabla)==len(resultado.boxes)
if not tabla.empty:
    primera=tabla.iloc[0]
    print('Primera detección:',primera.categoria,
          '· puntuación:',round(primera.puntuacion,3),
          '· tamaño:',round(primera.ancho_px),'×',round(primera.alto_px),'px')


## 7. Contar propuestas y revisar la foto

Si tres filas dicen persona, el registro tiene tres detecciones de esa categoría. El gráfico muestra ese conteo. No demuestra que existan exactamente tres personas: podría faltar una o repetirse una propuesta.

Si una categoría no aparece, hubo **cero propuestas conservadas** para ella, no prueba de que esté ausente. Revisa la foto y el catálogo de categorías antes de concluirlo.

**Tip:** suma las barras y compáralas con el total de filas. Deben coincidir porque resumen la misma salida. Contar aquí no sigue personas a través de un video ni reconoce su identidad.


In [ ]:
def resumir(t):
    return (t.groupby('categoria').size().rename('detecciones').reset_index()
             .sort_values('detecciones',ascending=False))
resumen=resumir(tabla)
display(resumen)
fig,ax=plt.subplots(figsize=(7,3))
if not resumen.empty:
    ax.bar(resumen.categoria,resumen.detecciones,color='#0075ad')
    ax.set_yticks(range(0,int(resumen.detecciones.max())+1))
else:ax.text(.5,.5,'Sin detecciones conservadas',ha='center',transform=ax.transAxes)
ax.set_ylabel('Detecciones en esta foto');ax.set_title('Un registro derivado de las cajas')
plt.tight_layout();plt.savefig(SALIDA/'04_conteo.png',dpi=130);plt.show()


## 8. Ver qué cambia al elevar el umbral
Ejecuta la comparación y revisa qué detecciones se conservan en cada umbral. Usamos la misma foto y los mismos pesos; solamente cambia el umbral.

Un umbral alto elimina detecciones con puntuación menor. Puede descartar errores, pero también objetos reales. **Más cajas no significa automáticamente mejor resultado.** Este conteo no es precision, recall ni mAP.


**De dónde sale el umbral:** es una regla elegida para conservar propuestas. Las puntuaciones provienen de pesos ya entrenados; el corte no se calcula como su promedio. Los tres valores de abajo son ensayos didácticos, no un umbral óptimo. Para elegir uno harían falta imágenes anotadas de validación y comparar omisiones y falsas detecciones; una prueba separada comprobaría después la elección. Contar menos cajas no demuestra mayor precisión. No confundas este corte con un criterio de coincidencia de cajas como IoU.

**Qué se exporta:** el bucle usa `r` para comparar; no reemplaza `tabla` ni `resumen`. Los CSV del bloque 11 corresponden al `UMBRAL` de la primera inferencia del bloque 5. Si cambias ese UMBRAL, vuelve a ejecutar desde allí hasta exportar.


In [ ]:
umbrales=[0.15,0.50,0.85]
comparacion=[]
fig,axs=plt.subplots(1,3,figsize=(12,6))
for ax,u in zip(axs,umbrales):
    r=modelo.predict(str(foto),conf=u,imgsz=640,device='cpu',verbose=False)[0]
    ax.imshow(r.plot()[:,:,::-1]);ax.axis('off')
    ax.set_title(f'Umbral {u:.2f}: {len(r.boxes)} detecciones')
    comparacion.append({'umbral':u,'detecciones':len(r.boxes)})
plt.tight_layout();plt.savefig(SALIDA/'05_umbrales.png',dpi=130);plt.show()
display(pd.DataFrame(comparacion))


## 9. Revisar solo una categoría y acercarnos a un objeto
En un parqueadero podríamos filtrar vehículos; en este ejemplo filtramos personas. **Filtrar no vuelve a entrenar** el modelo: solo conserva categorías conocidas.

Después recortamos una caja para entender sus coordenadas. Ese acercamiento no ejecuta un clasificador adicional ni reconoce la identidad de nadie.


In [ ]:
id_persona=next(i for i,n in modelo.names.items() if n=='person')
solo_personas=modelo.predict(str(foto),classes=[id_persona],conf=UMBRAL,
                             imgsz=640,device='cpu',verbose=False)[0]
display(tabla_detecciones(solo_personas).round(3))
if len(solo_personas.boxes):
    x1,y1,x2,y2=solo_personas.boxes.xyxy[0].cpu().numpy()
    recorte=original.crop((max(0,int(x1)),max(0,int(y1)),
                           min(original.width,int(x2)),min(original.height,int(y2))))
    fig,ax=plt.subplots(figsize=(3,4));ax.imshow(recorte);ax.axis('off')
    ax.set_title('Un recorte de la primera caja de persona');plt.tight_layout();plt.show()
else:print('No hay personas conservadas con este umbral.')


## 10. Otra foto: mismo procedimiento, otra entrada

Usamos zidane.jpg, también incluida en la biblioteca. No necesitas saber quién aparece: revisamos objetos de categorías conocidas, no identidades.

**Antes de ejecutar:** ¿qué conserva el procedimiento? Modelo, límite de puntuación y tamaño de entrada. ¿Qué cambia? Los píxeles de la foto y, por tanto, las propuestas.

**Después:** rastrea una caja y comprueba su fila. Dos fotos permiten revisar el mecanismo; no bastan para afirmar que funciona en otras cámaras, lugares o categorías. Esta segunda prueba no sustituye la foto y tabla que se exportarán en la sección 11.


In [ ]:
segunda=ASSETS/'zidane.jpg'
r2=modelo.predict(str(segunda),conf=UMBRAL,imgsz=640,device='cpu',verbose=False)[0]
tabla2=tabla_detecciones(r2)
display(resumir(tabla2))
fig,ax=plt.subplots(figsize=(9,5));ax.imshow(r2.plot()[:,:,::-1]);ax.axis('off')
ax.set_title('La misma herramienta, una entrada diferente');plt.tight_layout();plt.show()


### Los formatos del registro

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **CSV** — CSV: Comma-Separated Values (Valores separados por comas). Formato de texto que organiza registros con campos separados por comas. Los campos que contienen comas o saltos de línea pueden escribirse entre comillas; por eso un registro no siempre ocupa una sola línea visible. Puede incluir encabezados con nombres de columnas. Abrirlo como tabla no comprueba si sus datos o predicciones son correctos. [Fuente](https://www.rfc-editor.org/rfc/rfc4180)

- **JSON** — JSON: JavaScript Object Notation (Notación de objetos de JavaScript). Formato de texto que guarda datos estructurados con nombres, valores y listas. Aunque su nombre menciona JavaScript, otros lenguajes también pueden leerlo. Guardar la configuración ayuda a reconstruir una ejecución. [Fuente](https://www.rfc-editor.org/rfc/rfc8259)


## 11. Guarda un resultado que pueda revisarse

El código crea cuatro archivos: una imagen con las cajas, un CSV con una fila por propuesta, otro CSV con el conteo por categoría y un JSON con la configuración. CSV es un archivo de filas y columnas que puedes abrir en Excel o Sheets; JSON conserva nombres y valores para reconstruir cómo se obtuvo la salida.

Se guarda **la foto de la sección 4 y el UMBRAL de la primera inferencia de la sección 5**. Los ensayos de la sección 8 usan otra variable y no cambian automáticamente esos archivos. Si quieres guardar otra configuración, cambia UMBRAL y vuelve a ejecutar desde la sección 5 hasta esta sección.

**Tip:** abre la imagen marcada y verifica una fila antes de interpretar el archivo como registro. Descargar una tabla no convierte predicciones en inventario confirmado.


In [ ]:
tabla.to_csv(SALIDA/'detecciones.csv',index=False,encoding='utf-8-sig')
resumen.to_csv(SALIDA/'conteo.csv',index=False,encoding='utf-8-sig')
config={'modelo':'yolo26n.pt','ultralytics':ultralytics.__version__,
        'torch':torch.__version__,'foto':foto.name,'umbral':UMBRAL,'imgsz':640,
        'dispositivo':'cpu','entrenamiento_en_este_cuaderno':False}
(SALIDA/'configuracion.json').write_text(json.dumps(config,indent=2,ensure_ascii=False))
for p in [SALIDA/'02_foto_detectada.jpg',SALIDA/'detecciones.csv',SALIDA/'conteo.csv',SALIDA/'configuracion.json']:
    print('Archivo creado:',p)
    display(FileLink(str(p)))


## 12. Prueba con una foto propia (opcional)
Esta sección es opcional y no pide una entrega. Con `SUBIR_FOTO = True`, Colab abre un selector. Puedes subir una foto de objetos cotidianos de las categorías conocidas. También puedes indicar una ruta de archivo ya disponible.

Los archivos se procesan en el entorno donde ejecutas el cuaderno; no se envían a una API de predicción externa.


In [ ]:
SUBIR_FOTO = False
RUTA_FOTO_PROPIA = ''  # Ejemplo: '/content/mi_oficina.jpg'
if SUBIR_FOTO:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError('El selector funciona en Colab; fuera de Colab indica RUTA_FOTO_PROPIA.')
    cargados=files.upload()
    if cargados:RUTA_FOTO_PROPIA=next(iter(cargados))
if RUTA_FOTO_PROPIA:
    propia=Path(RUTA_FOTO_PROPIA)
    if not propia.is_file():raise FileNotFoundError(str(propia))
    Image.open(propia).verify()
    rp=modelo.predict(str(propia),conf=UMBRAL,imgsz=640,device='cpu',verbose=False)[0]
    tp=tabla_detecciones(rp);cp=resumir(tp)
    rp.save(filename=str(SALIDA/'mi_foto_detectada.jpg'))
    tp.to_csv(SALIDA/'mi_foto_detecciones.csv',index=False,encoding='utf-8-sig')
    cp.to_csv(SALIDA/'mi_foto_conteo.csv',index=False,encoding='utf-8-sig')
    display(cp)
    fig,ax=plt.subplots(figsize=(9,6));ax.imshow(rp.plot()[:,:,::-1]);ax.axis('off');plt.show()
else:print('Opción de foto propia desactivada: la demostración completa ya funcionó con dos fotos incluidas.')


### DL4: siglas para comprobar las propuestas

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **IoU** — IoU: Intersection over Union (Intersección sobre unión). Medida de superposición entre dos regiones o cajas: área compartida dividida entre el área cubierta por ambas. La unión cuenta la zona compartida una sola vez. No tiene unidades: 0 significa sin área común y 1, coincidencia completa; no es la puntuación del detector. [Fuente](https://docs.ultralytics.com/guides/yolo-performance-metrics)

- **TP** — TP: True Positive (Verdadero positivo). Caso positivo correctamente identificado como positivo. Primero se declara qué significa positivo. En detección requiere emparejar una propuesta válida con un objeto de referencia. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)

- **FP** — FP: False Positive (Falso positivo). Propuesta positiva que no corresponde a un positivo real. Es una falsa alerta. En detección cuenta una propuesta conservada sin coincidencia válida. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)

- **FN** — FN: False Negative (Falso negativo). Caso realmente positivo que no se identifica correctamente. Es una omisión. En detección cuenta un objeto de referencia sin propuesta válida emparejada. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)

- **AP** — AP: Average Precision (Precisión promedio). Resumen de la relación entre precisión y recall al recorrer los resultados ordenados por puntuación. Su cálculo exacto depende del protocolo de evaluación. En detección se fija el criterio de categoría, emparejamiento e IoU; AP no es la precisión en un único umbral. [Fuente](https://docs.ultralytics.com/guides/yolo-performance-metrics)

- **mAP** — mAP: mean Average Precision (Media de las precisiones promedio). Promedio de las AP de las categorías evaluadas según un protocolo. Algunas versiones también promedian resultados de varios criterios de IoU. Hay que indicar el protocolo: mAP50 usa IoU 0,50; mAP50–95 incluye IoU de 0,50 a 0,95 en pasos de 0,05. [Fuente](https://docs.ultralytics.com/modes/val)


## 13. Evaluación didáctica: comprobar cajas

**Encuentro 4:** las fotos anteriores no traen aquí respuestas anotadas. Para entender cómo se evalúa una caja, haremos un ejemplo pequeño con un objeto y rectángulos inventados. No usa las puntuaciones ni imágenes de YOLO y **no mide su mAP**.

Una **referencia** es la respuesta preparada para comprobar: qué objeto hay y dónde está. La caja de relleno azul es su posición. La caja de contorno naranja es la propuesta. Supondremos la misma categoría correcta en ambas; así aislamos la ubicación.

**Mira antes de calcular:** si movemos la propuesta 10 píxeles, comparte bastante superficie; si la movemos 30, comparte menos. IoU compara el área compartida con toda el área cubierta por ambas cajas: común / unión. Coincidencia completa da 1; sin zona común da 0.

Elegimos dos límites distintos:
- Puntuación: conserva propuestas con puntuación **mayor** que 0,50, igual que la convención documentada de esta biblioteca.
- IoU: acepta coincidencia si comparte al menos 0,50 del área de unión, y la categoría coincide.

Contamos **TP**, propuesta correcta emparejada con un objeto; **FP**, propuesta sin coincidencia válida; y **FN**, objeto de referencia sin detección correcta. Cada objeto puede dar crédito a una sola propuesta.

**Predice:** una propuesta mal situada cuenta como FP y deja el objeto como FN a la vez. Si la descartamos por puntuación, no queda una FP conservada, pero el objeto sigue sin detectar.


In [ ]:
# Datos inventados para aprender evaluación, independientes de las fotos y de resultado.
from matplotlib.patches import Rectangle
def iou_cajas(a,b):
    izquierda=max(a[0],b[0]);arriba=max(a[1],b[1])
    derecha=min(a[2],b[2]);abajo=min(a[3],b[3])
    comun=max(0,derecha-izquierda)*max(0,abajo-arriba)
    area_a=max(0,a[2]-a[0])*max(0,a[3]-a[1])
    area_b=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    union=area_a+area_b-comun
    return comun/union if union else 0.

referencia=np.array([10.,20.,50.,60.]) # ancho 40 y alto 40 píxeles
PUNTUACION_EJEMPLO=.80;MINIMO_IOU=.50
escenarios=[('Desplazar 10 px',10,.50),('Desplazar 30 px',30,.50),('Filtrar con 0,90',10,.90)]
fig,axs=plt.subplots(1,3,figsize=(12,4));filas=[]
for ax,(nombre,desplazamiento,limite_puntuacion) in zip(axs,escenarios):
    propuesta=referencia+np.array([desplazamiento,0,desplazamiento,0])
    iou=iou_cajas(referencia,propuesta)
    conservada=PUNTUACION_EJEMPLO>limite_puntuacion
    tp=int(conservada and iou>=MINIMO_IOU);fp=int(conservada and not tp);fn=1-tp
    ax.add_patch(Rectangle(referencia[:2],40,40,color='#0075ad',alpha=.3,label='Referencia'))
    ax.add_patch(Rectangle(propuesta[:2],40,40,fill=False,edgecolor='#b85d00',lw=2,
                           linestyle='-' if conservada else '--',label='Propuesta' if conservada else 'Descartada'))
    comun_x=max(referencia[0],propuesta[0]);comun_ancho=max(0,min(referencia[2],propuesta[2])-comun_x)
    if comun_ancho:ax.add_patch(Rectangle((comun_x,20),comun_ancho,40,color='#198a87',alpha=.4,label='Área común'))
    ax.set(xlim=(0,95),ylim=(80,0),xlabel='x: píxeles desde la izquierda',ylabel='y: píxeles desde arriba')
    ax.set_aspect('equal');ax.set_title(f'{nombre}\nIoU={iou:.3f} · TP/FP/FN={tp}/{fp}/{fn}',fontsize=10);ax.legend(fontsize=7)
    filas.append([iou,tp,fp,fn])
    print(nombre,'· puntuación',PUNTUACION_EJEMPLO,'· límite',limite_puntuacion,
          '· conservada',conservada,'· aciertos/falsas propuestas/omisiones:',tp,fp,fn)
plt.tight_layout();plt.savefig(SALIDA/'06_evaluacion_cajas.png',dpi=130);plt.show()
assert np.isclose(filas[0][0],.6) and np.isclose(filas[1][0],1/7)
if MINIMO_IOU==.50 and PUNTUACION_EJEMPLO==.80:
    assert filas[0][1:]==[1,0,0] and filas[1][1:]==[0,1,1] and filas[2][1:]==[0,0,1]
else:
    print('Configuración cambiada: interpreta los nuevos conteos con el criterio que elegiste.')
print('IoU desplazamiento 10: área común 30×40=1200; unión 1600+1600−1200=2000; 1200/2000=0,60.')
print('IoU desplazamiento 30: área común 10×40=400; unión 1600+1600−400=2800; 400/2800=1/7.')


### Después: explica los dos tipos de límite y los errores

- Desplazar 10 píxeles da IoU 0,60: un acierto y ninguna omisión.
- Desplazar 30 da aproximadamente 0,143: una propuesta incorrecta **y** un objeto omitido. No son categorías excluyentes de un mismo registro; contamos propuestas y objetos de referencia.
- Con filtro de puntuación 0,90, la propuesta de 0,80 se descarta. No hay propuesta conservada incorrecta, pero sigue habiendo un objeto omitido.

**Tip:** cambia solo MINIMO_IOU a 0,70 y ejecuta de nuevo. La caja no se mueve, pero el primer caso deja de cumplir la regla. Después restaura 0,50. El código mantiene las comprobaciones de áreas; los conteos se comprueban contra los valores iniciales solo cuando conservas la configuración inicial. Si cambias el criterio, explica por qué deben cambiar esos conteos.

**Precisión** pregunta qué parte de las propuestas conservadas fue correcta: TP/(TP+FP). **Recall** pregunta qué parte de los objetos de referencia se encontró: TP/(TP+FN). Si no hay propuestas, la precisión tiene denominador cero y no está definida: no hay que anunciar “100 %”.

Para obtener **AP** se repite la comparación recorriendo puntuaciones y se resume la relación precisión–recall según un protocolo. **mAP** promedia AP entre categorías o criterios definidos por ese protocolo. Los tres dibujos enseñan conteos; **no calculan AP ni mAP de YOLO**.


## Conecta lo que acabas de hacer con los encuentros

- **DL2 · Transferencia y visión:** una foto y un catálogo de categorías entran; unas cajas, nombres y puntuaciones salen. Usaste pesos previos sin entrenarlos para otras categorías.
- **DL4 · Detección y evaluación:** el filtro decide qué propuestas conservas; la referencia permite comprobar categoría y ubicación. La sección 13 explica por qué una caja falsa también puede dejar un objeto sin detectar.
- **DL4 · Decisión:** exportar organiza propuestas. Antes de usar un conteo como inventario, una persona debe revisar omisiones y duplicados. En el bloque 14 del laboratorio DL compararás políticas con costes supuestos; sus puntuaciones de granos son otro ejemplo, no salidas de estas fotos.

**Comprueba sin memorizar:** explica una caja visible, rastrea su registro, identifica de dónde salió el límite y di qué faltaría para evaluar con fotos representativas anotadas. No basta responder “salieron muchas cajas”.

**Si algo falla:** vuelve al primer mensaje de error y ejecuta las celdas previas del recorrido. Para descargar pesos se necesita internet. Con cero detecciones, comprueba categorías, foto y umbral antes de cambiar muchas cosas. Si un entorno trae bibliotecas incompatibles, reinícialo después de seguir su preparación; no actives recursos de pago para resolver un fallo de instalación.

Fuentes oficiales: [YOLO26](https://docs.ultralytics.com/models/yolo26/), [Predicción y resultados](https://docs.ultralytics.com/modes/predict/), [Detección](https://docs.ultralytics.com/tasks/detect/). La sección 13 es un ejemplo didáctico original con cajas inventadas. Para distribuir una aplicación, consulta [las condiciones del software y pesos](https://www.ultralytics.com/license).
